# 1 - Data Acquisition
This notebook handles the downloading and initialization of the benchmark datasets (`karate` and `dolphins`) used in the 10th DIMACS Implementation Challenge. These exact, small-scale graphs will serve as the ground-truth unit tests for our modularity maximization algorithm.

In [12]:
using Graphs
using MatrixDepot
using SimpleWeightedGraphs
using Random

## 1.1 - Zachary's Karate Club Network
The `karate` dataset is a classic social network graph consisting of 34 nodes and 78 edges. It is natively included within the `Graphs.jl` ecosystem.

In [14]:
karate_graph = smallgraph(:karate)

println("Karate network loaded successfully:")
println("- Number of nodes: ", nv(karate_graph))
println("- Number of edges: ", ne(karate_graph))

Karate network loaded successfully:
- Number of nodes: 34
- Number of edges: 78


## 1.2 - The Dolphins Social Network
The `dolphins` dataset contains 62 nodes and 159 edges. We use `MatrixDepot.jl` to automatically fetch the sparse adjacency matrix from the SuiteSparse/DIMACS public archive (indexed as "Newman/dolphins") and convert it into a standard undirected graph.

In [15]:
dolphins_matrix = matrixdepot("Newman/dolphins")

dolphins_graph = SimpleGraph(dolphins_matrix)

println("\nDolphins network loaded successfully:")
println("- Number of nodes: ", nv(dolphins_graph))
println("- Number of edges: ", ne(dolphins_graph))


Dolphins network loaded successfully:
- Number of nodes: 62
- Number of edges: 159


# 2 - Objective Function & Baseline Heuristic


## 2.1 - Objective function 

To compare candidate partitions for modularity maximization, we compute Newman-Girvan modularity.
This score measures how much more edge weight lies inside communities than would be expected by chance in a random graph with the same node strengths.

The implementation below is generic:
- unweighted graphs are treated as if every existing edge has weight `1.0`
- weighted graphs built with `SimpleWeightedGraphs.jl` use their actual edge weights
- the community assignment is given as a label vector indexed by vertex

This lets us evaluate both `Graph` and `SimpleWeightedGraph` inputs using the same objective function.

In [16]:
function modularity(g, community)
    n = nv(g)
    @assert length(community) == n "community vector must have one label per vertex"

    has_weight = hasmethod(weight, Tuple{typeof(g), Int, Int})
    edge_weight(u, v) = has_weight ? weight(g, u, v) : 1.0

    m = 0.0
    for e in edges(g)
        m += edge_weight(src(e), dst(e))
    end

    if m == 0.0
        return 0.0
    end

    degree_weight = zeros(Float64, n)
    for e in edges(g)
        u, v = src(e), dst(e)
        w = edge_weight(u, v)
        degree_weight[u] += w
        degree_weight[v] += w
    end

    Q = 0.0
    for e in edges(g)
        u, v = src(e), dst(e)
        if community[u] == community[v]
            w = edge_weight(u, v)
            Q += w - degree_weight[u] * degree_weight[v] / (2m)
        end
    end

    return Q / (2m)
end

modularity (generic function with 1 method)

In [17]:
example_comm = [1 for _ in vertices(karate_graph)]
println("Karate uniform-partition modularity: ", modularity(karate_graph, example_comm))

Karate uniform-partition modularity: 0.35042735042735046


## 2.2 - Baseline Heuristic


### 2.2.1 - Simple Label Propagation (LPA)

We implement a basic Label Propagation Algorithm inspired by Section 3 of the LPAm+ paper.
The procedure is intentionally simple:
1. every node is initially assigned a unique label;
2. at each iteration, each node adopts the most frequent label among its neighbors;
3. when there is a tie, one of the tied labels is chosen uniformly at random.

This version does not include modularity maximization or community merging, so the result can be unstable and may yield one or two communities depending on the random seed.

In [34]:
function lpa(g; max_iter=100, seed=1)
    rng = MersenneTwister(seed)
    n = nv(g)

    labels = collect(1:n)
    current_labels = copy(labels)

    for _ in 1:max_iter
        changed = false
        new_labels = copy(current_labels)

        for u in 1:n
            nbrs = [v for v in neighbors(g, u) if v != 0]
            if isempty(nbrs)
                continue
            end

            counts = Dict{Int, Int}()
            for v in nbrs
                lab = current_labels[v]
                counts[lab] = get(counts, lab, 0) + 1
            end

            max_count = maximum(values(counts))
            candidates = [lab for (lab, count) in counts if count == max_count]
            chosen_label = candidates[rand(rng, 1:length(candidates))]

            if new_labels[u] != chosen_label
                new_labels[u] = chosen_label
                changed = true
            end
        end

        current_labels = new_labels
        if !changed
            break
        end
    end

    return current_labels
end


lpa (generic function with 1 method)

In [35]:
karate_labels = lpa(karate_graph; seed=42)
println("Karate labels: ", karate_labels)
println("Number of communities: ", length(unique(karate_labels)))

Karate labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 15, 1, 1, 1, 1, 15, 15, 1, 1, 15, 1, 15, 1, 15, 15, 1, 15, 15, 15, 1, 15, 1, 15, 1, 1]
Number of communities: 2
